# Procesamiento de Lenguaje Natural a Consultas SQL (Text-to-SQL)

¡Hola! En este notebook vamos a construir un sistema muy chulo de **NLP a SQL**. El objetivo es que cualquier persona pueda escribir una pregunta en lenguaje natural (como *"¿Cuál fue la temperatura máxima en Madrid la semana pasada?"*) y nuestro sistema la traduzca automáticamente a una consulta SQL válida de PostgreSQL, la ejecute en nuestra base de datos en AWS RDS, y nos devuelva los datos limpios en un DataFrame de Pandas.

### Tecnologías que usaremos:
1. **LangChain**: Para orquestar la comunicación con el modelo de lenguaje de Google Gemini.
2. **Pydantic**: Para obligar al modelo a devolvernos la consulta SQL estructurada en un formato JSON limpio.
3. **psycopg (v3)**: La librería estándar de Python que venimos usando para conectarnos y ejecutar comandos en PostgreSQL RDS.
4. **Pandas**: Para manejar y visualizar el resultado de la base de datos de manera profesional.
5. **FastAPI** (Prototipo al final): Para ver cómo envolver todo esto en una API web real con Swagger UI interactivo.

---

## Paso 1: Carga de Dependencias y Variables de Entorno

Lo primero que necesitamos es importar todas las librerías necesarias y cargar nuestras claves de API desde el archivo `.env`. Recuerda que para usar Gemini necesitas configurar `GEMINI_API_KEY` (o `GOOGLE_API_KEY`) en tu archivo `.env`.

In [1]:
# -*- coding: utf-8 -*-
import os
import sys
import pandas as pd
import psycopg
from pathlib import Path
from dotenv import load_dotenv

# Importaciones para LangChain y Pydantic
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

print("Librerías importadas correctamente.")

# Buscamos y cargamos el archivo .env de la carpeta de Miguel
BASE_DIR = Path("__file__").resolve().parent
env_path = BASE_DIR / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print("Archivo .env cargado con éxito en el Notebook.")
else:
    load_dotenv()
    print("No se encontró .env local, buscando en variables del sistema.")

Librerías importadas correctamente.
Archivo .env cargado con éxito en el Notebook.


## Paso 2: Configuración de la Base de Datos RDS

Definimos los parámetros para conectarnos a la base de datos PostgreSQL alojada en AWS RDS. Estos datos son fijos para nuestro proyecto. Usaremos `psycopg` para abrir y cerrar conexiones de forma segura.

In [2]:
# Datos de conexión con RDS proporcionados para el proyecto AEMET
HOST_RDS = "database-dai07rt-proyecto-aemet-2026.cr828ecqk1a6.eu-central-1.rds.amazonaws.com"
PORT_RDS = "5432"
DBNAME_RDS = "postgres"
USER_RDS = "aemet2026"
PASSWORD_RDS = "mondongo-dai07rt-aemet-2026"

# Construimos el DSN (Data Source Name) para psycopg con connect_timeout de 10 segundos
dsn_conexion = f"host={HOST_RDS} port={PORT_RDS} dbname={DBNAME_RDS} user={USER_RDS} password={PASSWORD_RDS} connect_timeout=10"

def probar_conexion_rds():
    """
    Intenta conectarse al servidor RDS y comprueba que responde bien.
    Muestra un print con el resultado o el error si falla (por ejemplo, por temas de IPs o firewall).
    """
    print("Probando la conexión con la base de datos de AWS RDS...")
    try:
        with psycopg.connect(dsn_conexion) as conexion:
            with conexion.cursor() as cursor:
                cursor.execute("SELECT version();")
                version = cursor.fetchone()
                print("¡Conexión Exitosa con PostgreSQL RDS!")
                print(f"Versión del servidor: {version[0]}")
                return True
    except Exception as e:
        print(f"Error al conectar con la base de datos RDS: {e}")
        print("\n-> NOTA: Si el error es de timeout o de red, revisa si tu IP pública está añadida al Grupo de Seguridad de AWS (Security Group) en el puerto 5432.")
        return False

# Probamos la conexión
conexion_ok = probar_conexion_rds()

Probando la conexión con la base de datos de AWS RDS...
¡Conexión Exitosa con PostgreSQL RDS!
Versión del servidor: PostgreSQL 18.3 on aarch64-unknown-linux-gnu, compiled by aarch64-unknown-linux-gnu-gcc (GCC) 12.4.0, 64-bit


## Paso 3: Definición del Esquema DDL para el LLM

Para que Gemini pueda redactar consultas SQL correctas y sin equivocarse en los nombres de las columnas, necesitamos darle la estructura exacta (DDL) de nuestras tablas de la base de datos.

In [3]:
# Definimos el esquema DDL exacto de la base de datos como un string multilinea.
# Se lo pasaremos al LLM en el prompt del sistema.
ESQUEMA_BD = """
-- Tabla 1: Catálogo Maestro de Estaciones (Catálogo de ubicaciones)
CREATE TABLE estaciones (
    indicativo VARCHAR(10) PRIMARY KEY, -- Identificador único de la estación (ej. '3195', 'B228')
    nombre VARCHAR(150),                -- Nombre completo de la estación meteorológica
    provincia VARCHAR(100),             -- Provincia donde se ubica la estación
    latitud FLOAT,                      -- Latitud geográfica
    longitud FLOAT,                     -- Longitud geográfica
    altitud INT,                        -- Altitud sobre el nivel del mar en metros
    indsinop VARCHAR(10)                -- Indicativo sinóptico internacional
);

-- Tabla 2: Historial de Mediciones Diarias (Datos climáticos acumulados día a día)
CREATE TABLE datos_climaticos (
    fecha DATE,                         -- Fecha de la medición (YYYY-MM-DD)
    indicativo VARCHAR(10) REFERENCES estaciones(indicativo), -- Relación con la tabla estaciones
    nombre VARCHAR(150),                -- Nombre de la estación en el momento de la medida
    provincia VARCHAR(255),             -- Provincia de la medida
    altitud INT,                        -- Altitud registrada en metros
    tmed FLOAT,                         -- Temperatura media del día en ºC
    prec FLOAT,                         -- Precipitación/Lluvia acumulada en el día en mm (l/m2)
    tmin FLOAT,                         -- Temperatura mínima registrada en el día en ºC
    horatmin TIME,                      -- Hora a la que se registró la tmin
    tmax FLOAT,                         -- Temperatura máxima registrada en el día en ºC
    horatmax TIME,                      -- Hora a la que se registró la tmax
    dir FLOAT,                          -- Dirección del viento en decenas de grado
    velmedia FLOAT,                     -- Velocidad media del viento en m/s
    racha FLOAT,                        -- Racha máxima del viento en m/s
    horaracha VARCHAR(15),              -- Hora o intervalo de la racha máxima
    sol FLOAT,                          -- Horas de sol registradas en el día
    presMax FLOAT,                      -- Presión atmosférica máxima en hPa
    horaPresMax TIME,                   -- Hora de la presión máxima
    presMin FLOAT,                      -- Presión atmosférica mínima en hPa
    horaPresMin TIME,                   -- Hora de la presión mínima
    hrMedia FLOAT,                      -- Humedad relativa media del día en %
    hrMax FLOAT,                        -- Humedad relativa máxima del día en %
    horaHrMax TIME,                     -- Hora de la humedad máxima
    hrMin FLOAT,                        -- Humedad relativa mínima del día en %
    horaHrMin TIME,                     -- Hora de la humedad mínima
    PRIMARY KEY (indicativo, fecha)
);
"""
print("Esquema DDL preparado para ser inyectado al LLM.")

Esquema DDL preparado para ser inyectado al LLM.


## Paso 4: Pydantic para Estructurar la Salida de Gemini

Queremos que el LLM no nos devuelva texto libre con explicaciones largas, sino una estructura muy limpia. Crearemos una clase usando Pydantic. 
Esto le dice a LangChain que queremos forzar al modelo a devolvernos un objeto con los atributos que definamos.

In [4]:
class ConsultaSQLGenerada(BaseModel):
    """Estructura de datos que debe devolver el LLM obligatoriamente."""
    query_sql: str = Field(
        description="Consulta SQL SELECT válida para PostgreSQL. Debe utilizar únicamente las tablas y campos del esquema provisto, sin inventar nombres."
    )
    explicacion: str = Field(
        description="Explicación sencilla y en español de qué hace la consulta SQL y cómo responde a la pregunta original del usuario."
    )

print("Estructura Pydantic declarada correctamente.")

Estructura Pydantic declarada correctamente.


## Paso 5: Conexión con Gemini y Definición del Prompt de LangChain

Inicializamos el modelo de chat usando la API de Gemini (a través de `langchain-google-genai`). Configuramos la temperatura a `0.0` porque queremos que las respuestas sean consistentes y lógicas al generar código SQL.

Para alinearnos con los requisitos del proyecto, utilizaremos el modelo **Gemini 3.1 Flash Lite** (`gemini-3.1-flash-lite`), optimizado para baja latencia y alta eficiencia en tareas de Text-to-SQL.

In [5]:
# Comprobamos que exista la API KEY de Gemini antes de instanciar el modelo
api_key_gemini = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key_gemini:
    print("⚠️ ADVERTENCIA: No se detectó GEMINI_API_KEY en las variables de entorno.")
    print("Por favor, añade 'GEMINI_API_KEY=tu_clave_de_google_ai_studio' al archivo .env o configúrala en el sistema.")
else:
    print("API Key de Gemini detectada.")

# Inicializamos el LLM. Usamos 'gemini-3.1-flash-lite' según definición del proyecto.
# Puedes cambiarlo por otros modelos de la gama si fuese necesario.
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)

# Vinculamos la salida estructurada de Pydantic al LLM
llm_estructurado = llm.with_structured_output(ConsultaSQLGenerada)

# Creamos la plantilla del Prompt
prompt_plantilla = ChatPromptTemplate.from_messages([
    ("system", """Eres un experto en bases de datos PostgreSQL de un equipo de analistas de datos.
Tu tarea es recibir una pregunta del usuario en español y traducirla a una consulta SQL SELECT correcta.

Utiliza únicamente el siguiente esquema DDL de la base de datos para construir la consulta SQL:
---------------------
{esquema_bd}
---------------------

Instrucciones críticas:
1. Devuelve una consulta SQL válida que se pueda ejecutar directamente en PostgreSQL.
2. No agregues comillas inclinadas (```sql) dentro del campo de la query en el JSON final.
3. Si la pregunta requiere buscar por provincia, realiza búsquedas utilizando LIKE de forma insensible a mayúsculas/minúsculas (ej: ILIKE '%madrid%') para evitar fallos de escritura.
4. Limita los resultados a 100 filas a menos que el usuario pida más.
5. Asegúrate de hacer JOINs correctos entre estaciones y datos_climaticos usando el campo 'indicativo' when sea necesario.
6. Explicale al usuario el proceso de la query de manera amigable en español en el campo explicacion.
"""),
    ("human", "Pregunta del usuario: {pregunta_usuario}")
])

# Construimos la cadena de ejecución de LangChain
cadena_text_to_sql = prompt_plantilla | llm_estructurado
print("Cadena de LangChain (Prompt + LLM con estructura Pydantic) configurada con éxito.")

API Key de Gemini detectada.
Cadena de LangChain (Prompt + LLM con estructura Pydantic) configurada con éxito.


## Paso 6: Orquestador Principal - NLP a SQL

Creamos la función principal `preguntar_a_la_bd`. Esta función llamará a nuestro LLM para generar la query SQL a partir de la pregunta del usuario, la mostrará por pantalla, la ejecutará en RDS usando un DataFrame de Pandas, y nos devolverá la información estructurada.

In [6]:
def preguntar_a_la_bd(pregunta):
    """
    Traduce la pregunta del usuario a SQL, la ejecuta en RDS y muestra el DataFrame resultante.
    
    Args:
        pregunta (str): Pregunta en lenguaje natural en español.
    Returns:
        pd.DataFrame: DataFrame de Pandas con los resultados. None si ocurre un error.
    """
    print(f"\n--- Procesando pregunta: '{pregunta}' ---")
    
    # 1. Llamada a Gemini usando LangChain para generar la query
    try:
        respuesta = cadena_text_to_sql.invoke({
            "esquema_bd": ESQUEMA_BD,
            "pregunta_usuario": pregunta
        })
    except Exception as e:
        print(f"Error al invocar el LLM Gemini: {e}")
        return None
    
    sql_generado = respuesta.query_sql
    explicacion = respuesta.explicacion
    
    print(f"\n[LLM] Explicación: {explicacion}")
    print(f"[LLM] Consulta SQL Generada:\n{sql_generado}\n")
    
    # 2. Conectamos con RDS y ejecutamos la consulta con Pandas
    print("Conectando a RDS para obtener los resultados...")
    try:
        # Usamos psycopg de forma directa para crear una conexión
        # y pd.read_sql para volcar los resultados en un DataFrame limpio
        with psycopg.connect(dsn_conexion) as conexion:
            df_resultado = pd.read_sql(sql_generado, conexion)
            
            print(f"¡Éxito! Obtenidos {len(df_resultado)} registros.")
            return df_resultado
            
    except Exception as e:
        print(f"Error al ejecutar la query SQL en RDS: {e}")
        return None

## Paso 7: Pruebas de Funcionamiento Básicas

Ejecución directa en el notebook utilizando la función `preguntar_a_la_bd` para validar el correcto funcionamiento.

In [7]:
# Prueba rápida de estaciones meteorológicas en Madrid
df_1 = preguntar_a_la_bd("Muestra las primeras 5 estaciones meteorológicas en la provincia de Madrid")
if df_1 is not None:
    display(df_1.head())
else:
    print("No se pudieron cargar datos de prueba.")


--- Procesando pregunta: 'Muestra las primeras 5 estaciones meteorológicas en la provincia de Madrid' ---

[LLM] Explicación: He realizado una consulta a la tabla 'estaciones' filtrando por la provincia de 'Madrid' utilizando ILIKE para asegurar que no haya problemas con las mayúsculas o minúsculas, y he limitado el resultado a las primeras 5 estaciones encontradas.
[LLM] Consulta SQL Generada:
SELECT * FROM estaciones WHERE provincia ILIKE '%madrid%' LIMIT 5

Conectando a RDS para obtener los resultados...


/tmp/ipykernel_18721/2842462435.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_resultado = pd.read_sql(sql_generado, conexion)


¡Éxito! Obtenidos 5 registros.


,indicativo,nombre,provincia,latitud,longitud,altitud,indsinop
0,2462,PUERTO DE NAVACERRADA,MADRID,40.793335,-4.010833,1892,08215
1,3100B,ARANJUEZ,MADRID,40.067223,-3.546111,540,08229
2,3104Y,RASCAFRÍA,MADRID,40.889721,-3.888333,1159,
3,3110C,BUITRAGO DEL LOZOYA,MADRID,41.006943,-3.613889,1030,08146
4,3111D,SOMOSIERRA,MADRID,41.135555,-3.580555,1450,08145


## Paso 8: Creación del Endpoint con FastAPI (Prototipo para Fase 2)

Para implementar el endpoint `/ask` de la Fase 2 (que conectará nuestra interfaz Streamlit con el backend), podemos portar la lógica anterior a una API web muy sencilla con FastAPI.

Para evitar que la celda de Jupyter termine y poder usar Swagger UI en tu navegador de forma persistente, levantaremos el servidor web en un hilo secundario y mantendremos activa la celda en primer plano con un bucle infinito.

In [8]:
import threading
import time
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

# Definimos el esquema del cuerpo que recibirá el endpoint
class PreguntaRequest(BaseModel):
    pregunta: str

# Inicializamos la aplicación FastAPI
app = FastAPI(
    title="API de Consulta Climatológica AEMET NLP",
    description="API para traducir lenguaje natural a consultas SQL y ejecutarlas en RDS."
)

@app.get("/")
def read_root():
    return {"status": "ok", "mensaje": "API Climatológica AEMET activa"}

@app.post("/ask")
async def ask_endpoint(payload: PreguntaRequest):
    """
    Endpoint que recibe una pregunta, llama al LLM, ejecuta la query en RDS
    y devuelve el resultado formateado en JSON junto al SQL generado.
    """
    pregunta = payload.pregunta
    if not pregunta.strip():
        raise HTTPException(status_code=400, detail="La pregunta no puede estar vacía.")
    
    # 1. Llamamos a la API de Gemini
    try:
        respuesta_llm = cadena_text_to_sql.invoke({
            "esquema_bd": ESQUEMA_BD,
            "pregunta_usuario": pregunta
        })
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error en LLM Gemini: {str(e)}")
    
    sql = respuesta_llm.query_sql
    explicacion = respuesta_llm.explicacion
    
    # 2. Ejecutamos en la BD de AWS
    try:
        with psycopg.connect(dsn_conexion) as conexion:
            df = pd.read_sql(sql, conexion)
            
            # Convertimos el DataFrame a diccionario para mandarlo por JSON
            registros = df.to_dict(orient="records")
            
            return {
                "pregunta": pregunta,
                "sql_generado": sql,
                "explicacion": explicacion,
                "registros_encontrados": len(registros),
                "datos": registros
            }
    except Exception as e:
        # Si da fallo el SQL (por ejemplo por un campo mal generado)
        # devolvemos error 422 indicando el SQL fallido para debuggear
        raise HTTPException(
            status_code=422,
            detail={
                "error": f"Error de sintaxis o ejecución de la query: {str(e)}",
                "sql_intentado": sql,
                "explicacion_llm": explicacion
            }
        )

def arrancar_api():
    """Arranca el servidor uvicorn."""
    try:
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
    except Exception as e:
        print(f"Error al iniciar uvicorn: {e}")

# Iniciamos el servidor en un hilo separado de fondo
hilo_api = threading.Thread(target=arrancar_api, daemon=True)
hilo_api.start()
print("===========================================================")
print("Servidor FastAPI iniciado en segundo plano en http://localhost:8000")
print("Accede a http://localhost:8000/docs en tu navegador para interactuar con Swagger UI.")
print("===========================================================")

# Bucle infinito para mantener viva la celda y permitir su uso persistente
try:
    print("Celda en ejecución persistente. Para apagar el servidor, pulsa 'Stop' (interrumpir kernel) en Jupyter.")
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nInterrupción detectada. Apagando servidor de forma segura...")


INFO:     Started server process [18721]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Servidor FastAPI iniciado en segundo plano en http://localhost:8000
Accede a http://localhost:8000/docs en tu navegador para interactuar con Swagger UI.
Celda en ejecución persistente. Para apagar el servidor, pulsa 'Stop' (interrumpir kernel) en Jupyter.
INFO:     127.0.0.1:53290 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:53290 - "GET /openapi.json HTTP/1.1" 200 OK


/tmp/ipykernel_18721/1164514054.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


INFO:     127.0.0.1:46976 - "POST /ask HTTP/1.1" 200 OK


/tmp/ipykernel_18721/1164514054.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


INFO:     127.0.0.1:49502 - "POST /ask HTTP/1.1" 200 OK

Interrupción detectada. Apagando servidor de forma segura...
